In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
import sys
print(sys.executable)

/home/fancy/venvs/woc312/bin/python


In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )



In [18]:
# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['tyiannak/pyAudioAnalysis','FZJ-INM1-BDA/siibra-explorer','spacetelescope/jwql', 'smhyatt/deforest-watch-with-BFAST-in-CUDA', 'eurostat/statistics-coded', 'tnagler/VineCopula', 'deeptime-ml/deeptime', 'aiidateam/aiida-tutorials', 'lepy/phuzzy', 'dpiparo/vdt']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df

,sha1,project
0,000a5c520d3dcf9cb1847f17092cb178032013f8,tyiannak_pyaudioanalysis
1,00204ff33b60af2b6da7b3f0ff82865fdcb971d1,tyiannak_pyaudioanalysis
2,0030b718e92eb6b4798f7fe0bef8b9757aad54e3,tyiannak_pyaudioanalysis
3,0046c875f107159789cafd14cf07d04f3ac195c4,tyiannak_pyaudioanalysis
4,0082170c5623bbcb7aa4cf5be46d82d67bc23943,tyiannak_pyaudioanalysis
...,...,...
310,fdb34ff87169659f67cd5c869b3dc21eb19767d1,dpiparo_vdt
311,fddf8aebb3269db1d8a3109eaaa3b1c88de382d7,dpiparo_vdt
312,fdea1cfcea097a09ddc362ab1ff21b10899fbad9,dpiparo_vdt
313,fe4efdcdaa8663193132cca1120ec55b12e4702e,dpiparo_vdt


In [19]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)

df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 29%|██▉       | 643/2184 [11:15<26:56,  1.05s/it]

Got Errors {'11357dd6767ea3eea1136e98333fdf75b74fb2de': 'Key 11357dd6767ea3eea1136e98333fdf75b74fb2de not found in /da5_fast/All.sha1c/commit_17.tch'}


 30%|███       | 661/2184 [11:33<26:37,  1.05s/it]

Got Errors {'16b9cbeae06c0000ade73ced2291d67d9e61203c': 'Key 16b9cbeae06c0000ade73ced2291d67d9e61203c not found in /da5_fast/All.sha1c/commit_22.tch'}


 33%|███▎      | 724/2184 [12:40<25:30,  1.05s/it]

...

Got Errors {'81d6f0036687f170609fe29091d51c5629920e14': 'Key 81d6f0036687f170609fe29091d51c5629920e14 not found in /da5_fast/All.sha1c/commit_1.tch'}


 98%|█████████▊| 2146/2184 [37:52<00:40,  1.07s/it]

Got Errors {'df448cd56c406f69df49c41f60a298a9553874b3': 'Key df448cd56c406f69df49c41f60a298a9553874b3 not found in /da5_fast/All.sha1c/commit_95.tch'}


100%|██████████| 2184/2184 [38:32<00:00,  1.06s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000a5c520d3dcf9cb1847f17092cb178032013f8,8c428776f1a407f5891c40140b0895410e75c0a6,[0dfe4059ec3675d50f6ea46156170491bedc7d13],Theodoros Giannakopoulos <tyiannak@gmail.com>,1411566537,+0300,Theodoros Giannakopoulos <tyiannak@gmail.com>,1411566537,+0300,added arff output in features\n,000a5c520d3dcf9cb1847f17092cb178032013f8,tyiannak_pyaudioanalysis
1,00204ff33b60af2b6da7b3f0ff82865fdcb971d1,4d2443e49d96c7e2dc5956493a85bb56e16cb166,[abd33bbe25ef3d5869da6e0f717e45163660e0c6],Theodoros Giannakopoulos <tyiannak@gmail.com>,1432738598,+0300,Theodoros Giannakopoulos <tyiannak@gmail.com>,1432738598,+0300,movie added\n,00204ff33b60af2b6da7b3f0ff82865fdcb971d1,tyiannak_pyaudioanalysis


In [20]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '000a5c520d3dcf9cb1847f17092cb178032013f8', 'tree': '8c428776f1a407f5891c40140b0895410e75c0a6', 'parent': ['0dfe4059ec3675d50f6ea46156170491bedc7d13'], 'author': 'Theodoros Giannakopoulos <tyiannak@gmail.com>', 'author_time': 1411566537, 'author_tz': '+0300', 'committer': 'Theodoros Giannakopoulos <tyiannak@gmail.com>', 'committer_time': 1411566537, 'committer_tz': '+0300', 'message': 'added arff output in features\n'}


In [21]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [22]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,dpiparo_vdt,000a5c520d3dcf9cb1847f17092cb178032013f8,Theodoros Giannakopoulos <tyiannak@gmail.com>,1411566537,added arff output in features\n


In [23]:
#mode a means append, so you have all your projects in the same file
yournetid='hnaicker'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [ ]:
#Code to get max and min times
df_commit_data['dt'] = pd.to_datetime(df_commit_data['author_time'], unit='s', utc=True)

time_range = df_commit_data.groupby('project')['dt'].agg(
    min_time='min',
    max_time='max',
).reset_index()

time_range

,project,min_time,max_time
0,aiidateam_aiida-tutorials,2018-05-28 16:53:57+00:00,2025-10-02 11:28:57+00:00
1,deeptime-ml_deeptime,2015-04-11 10:31:18+00:00,2025-04-14 02:31:18+00:00
2,dpiparo_vdt,2012-08-23 07:16:32+00:00,2025-05-03 22:18:58+00:00
3,eurostat_statistics-coded,2019-01-11 17:03:26+00:00,2025-03-20 13:04:12+00:00
4,fzj-inm1-bda_siibra-explorer,2017-07-24 08:51:16+00:00,2025-10-27 13:07:07+00:00
5,lepy_phuzzy,2017-06-28 07:38:13+00:00,2025-12-31 11:16:57+00:00
6,smhyatt_deforest-watch-with-bfast-in-cuda,2019-09-02 15:10:41+00:00,2025-03-18 17:41:59+00:00
7,spacetelescope_jwql,2017-11-13 16:39:24+00:00,2025-11-01 06:00:04+00:00
8,tnagler_vinecopula,2015-09-23 19:33:27+00:00,2025-08-07 13:35:26+00:00
9,tyiannak_pyaudioanalysis,2014-08-27 12:44:23+00:00,2025-08-18 14:51:43+00:00


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

FORMAT:

Git link

[number of stars, number of forks, last commit date]

[number of commits, number of authors, max time, min time]


https://github.com/tyiannak/pyAudioAnalysis :

[6.3k, 1.2k, 8/04/2025]

[782, 22, 2025-08-18 14:51:43+00:00, 2014-08-27 12:44:23+00:00]


https://github.com/FZJ-INM1-BDA/siibra-explorer :

[30, 6, 07/26/2026]

[3728, 6, 2025-10-27 13:07:07+00:00, 2017-07-24 08:51:16+00:00]

https://github.com/spacetelescope/jwql :

[68, 47, 08/15/2026]

[7880, 26, 2025-11-01 06:00:04+00:00, 2017-11-13 16:39:24+00:00]

https://github.com/smhyatt/deforest-watch-with-BFAST-in-CUDA :

[0, 0, 03/18/2025]

[643, 2, 2025-03-18 17:41:59+00:00, 2019-09-02 15:10:41+00:00]

https://github.com/eurostat/statistics-coded :

[42, 16, 03/20/2025]

[527, 7, 2025-03-20 13:04:12+00:00, 2019-01-11 17:03:26+00:00]

https://github.com/tnagler/VineCopula :

[101, 34, 08/28/2026]

[562, 9, 2025-08-07 13:35:26+00:00, 2015-09-23 19:33:27+00:00]

https://github.com/deeptime-ml/deeptime :

[890, 89, 06/26/2026]

[1319, 17, 2025-04-14 02:31:18+00:00, 2015-04-11 10:31:18+00:00]

https://github.com/aiidateam/aiida-tutorials :

[25, 37, 10/02/2025]

[573, 31, 2025-10-02 11:28:57+00:00, 2018-05-28 16:53:57+00:00]

https://github.com/lepy/phuzzy :

[3, 2, 07/17/2026]

[267, 3, 2025-12-31 11:16:57+00:00, 2017-06-28 07:38:13+00:00]

https://github.com/dpiparo/vdt :

[61, 18, 04/03/2025]

[164, 6, 2025-05-03 22:18:58+00:00, 2012-08-23 07:16:32+00:00]